# 2. Annotation Pipeline - Context-Aware Email Priority Labeling

**Project:** INTELIPS - Intelligent Email Priority System

**Author:** Karim Semaan

**Date:** November 20, 2024

---

## Objectives
1. Create annotation prompts for Groq API
2. Generate contextual scenarios (busy/quiet, time-of-day)
3. Annotate 2,000-5,000 email-context pairs with priority labels
4. Validate annotation quality with Cohen's Kappa
5. Create final annotated dataset for modeling

## Priority Levels
- **1 (Low):** Informational, no specific action required, can be deferred
- **2 (Normal):** Standard business communication, requires timely but not immediate response
- **3 (Critical):** Immediate action required, blocks other work, severe consequences if delayed

In [21]:
# Import libraries
import pandas as pd
import numpy as np
from groq import Groq
import os
from dotenv import load_dotenv
import json
import time
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Initialize Groq API client
groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")

groq_client = Groq(api_key=groq_api_key)

print("✓ Libraries imported")
print("✓ Groq API client initialized")
print("✓ Using OpenAI GPT-OSS-120B (120B params, 500+ tok/s, ultra-cheap!)")

✓ Libraries imported
✓ Groq API client initialized
✓ Using OpenAI GPT-OSS-120B (120B params, 500+ tok/s, ultra-cheap!)


## 2.1 Load Processed Emails

In [22]:
# Load processed emails from previous notebook
df = pd.read_csv('enron_processed_10k.csv')

print(f"Loaded {len(df):,} processed emails")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 10,000 processed emails
Columns: ['file', 'datetime', 'from', 'to', 'subject', 'body', 'hour', 'day_of_week', 'subject_length', 'body_length', 'recipient_count', 'has_urgency_in_subject', 'has_urgency_in_body']


,file,datetime,from,to,subject,body,hour,day_of_week,subject_length,body_length,recipient_count,has_urgency_in_subject,has_urgency_in_body
0,allen-p/_sent_mail/1.,2001-05-14 23:39:00+00:00,phillip.allen@enron.com,tim.belden@enron.com,NaN,Here is our forecast\n\n,23,0,0,23,1,False,False
1,allen-p/_sent_mail/10.,2001-05-04 20:51:00+00:00,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,20,4,3,786,1,False,False
2,allen-p/_sent_mail/100.,2000-10-18 10:00:00+00:00,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,10,2,8,30,1,False,False
3,allen-p/_sent_mail/1000.,2000-10-23 13:13:00+00:00,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Randy,\n\n Can you send me a schedule of the s...",13,0,0,187,1,False,False
4,allen-p/_sent_mail/1001.,2000-08-31 12:07:00+00:00,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,12,3,9,35,1,False,False


## 2.2 Create Contextual Scenarios

Following instructor feedback to cluster timestamps, we'll create workload states based on email volume patterns.

In [23]:
def generate_workload_state(row):
    """
    Generate workload state based on temporal features.
    Uses hour of day and day of week to simulate busy/quiet periods.
    """
    hour = row['hour']
    day_of_week = row['day_of_week']
    
    # Business hours (9-17) on weekdays are busier
    is_business_hours = (9 <= hour <= 17) and (day_of_week < 5)
    
    # Peak hours (14-16) are busiest
    is_peak_hours = (14 <= hour <= 16) and (day_of_week < 5)
    
    # Early morning / late evening / weekends are quiet
    is_off_hours = (hour < 7 or hour > 19) or (day_of_week >= 5)
    
    if is_peak_hours:
        return 'very_busy'
    elif is_business_hours:
        return 'busy'
    elif is_off_hours:
        return 'quiet'
    else:
        return 'normal'


def get_time_segment_description(hour):
    """Get descriptive time segment."""
    if 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 24:
        return 'evening'
    else:
        return 'night'


# Add contextual features
df['workload_state'] = df.apply(generate_workload_state, axis=1)
df['time_segment'] = df['hour'].apply(get_time_segment_description)

print("Workload state distribution:")
print(df['workload_state'].value_counts())

print("\nTime segment distribution:")
print(df['time_segment'].value_counts())

Workload state distribution:
workload_state
busy         3526
very_busy    2456
quiet        2227
normal       1791
Name: count, dtype: int64

Time segment distribution:
time_segment
afternoon    4819
morning      2506
evening      2194
night         481
Name: count, dtype: int64


## 2.3 Sample Emails for Annotation

Select diverse sample ensuring representation across different contexts.

In [24]:
def stratified_sample(df, n_samples=3000):
    """
    Create stratified sample ensuring diversity across:
    - Workload states
    - Time segments
    - Urgency indicators
    """
    samples = []
    
    # Sample from each workload state
    for workload in df['workload_state'].unique():
        workload_df = df[df['workload_state'] == workload]
        n_workload = int(n_samples * (len(workload_df) / len(df)))
        
        # Within each workload, ensure urgency diversity
        n_urgent = int(n_workload * 0.2)  # 20% with urgency keywords
        n_normal = n_workload - n_urgent
        
        urgent_sample = workload_df[workload_df['has_urgency_in_subject'] | workload_df['has_urgency_in_body']].sample(
            n=min(n_urgent, len(workload_df[workload_df['has_urgency_in_subject'] | workload_df['has_urgency_in_body']])),
            random_state=42
        )
        
        normal_sample = workload_df[~(workload_df['has_urgency_in_subject'] | workload_df['has_urgency_in_body'])].sample(
            n=min(n_normal, len(workload_df[~(workload_df['has_urgency_in_subject'] | workload_df['has_urgency_in_body'])])),
            random_state=42
        )
        
        samples.append(urgent_sample)
        samples.append(normal_sample)
    
    return pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)


# Create annotation sample
TARGET_ANNOTATIONS = 3000  # Start with 3000, can expand to 5000
df_annotate = stratified_sample(df, n_samples=TARGET_ANNOTATIONS)

print(f"Selected {len(df_annotate):,} emails for annotation")
print(f"\nWorkload distribution:")
print(df_annotate['workload_state'].value_counts())
print(f"\nUrgency indicator distribution:")
print(f"With urgency: {(df_annotate['has_urgency_in_subject'] | df_annotate['has_urgency_in_body']).sum():,}")
print(f"Without urgency: {(~(df_annotate['has_urgency_in_subject'] | df_annotate['has_urgency_in_body'])).sum():,}")

Selected 2,998 emails for annotation

Workload distribution:
workload_state
busy         1057
very_busy     736
quiet         668
normal        537
Name: count, dtype: int64

Urgency indicator distribution:
With urgency: 598
Without urgency: 2,400


## 2.4 Create Annotation Prompt

Design prompt that includes email content AND contextual factors.

In [25]:
def create_annotation_prompt(email_data):
    """
    Create annotation prompt with email content and context.
    """
    # Clean subject and body
    subject = str(email_data['subject']).strip()[:200]
    body = str(email_data['body']).strip()[:1000]  # Limit for API
    workload = email_data['workload_state']
    time_seg = email_data['time_segment']
    recipients = int(email_data['recipient_count']) if not pd.isna(email_data['recipient_count']) else 1
    
    # Map workload to description
    workload_desc = {
        'very_busy': 'extremely busy with back-to-back meetings and urgent tasks',
        'busy': 'moderately busy with ongoing tasks and regular meetings',
        'normal': 'normal workload with manageable tasks',
        'quiet': 'light workload with few pressing matters'
    }
    
    prompt = f"""You are an expert email prioritization assistant. Your task is to assign a priority level to an email based on its content AND the recipient's current context.

**Priority Levels:**
1 (Low): Informational content, no specific action required, can be deferred without consequences
2 (Normal): Standard business communication requiring timely but not immediate response (within hours/days)
3 (Critical): Immediate action required, blocks other work, or has severe consequences if delayed (within minutes/hour)

**Recipient Context:**
- Current workload: {workload_desc.get(workload, 'normal')}
- Time of day: {time_seg}
- Number of recipients: {recipients}

**Email Details:**
Subject: {subject if subject else "(no subject)"}

Body:
{body if body else "(no body)"}....

**Instructions:**
1. Consider BOTH the email content (urgency keywords, deadlines, action items) AND the recipient's context
2. In a busy workload, even normally important emails might be deprioritized unless truly urgent
3. In a quiet period, emails requiring timely response get more attention
4. Emails with many recipients might be less urgent for any single person
5. Respond with ONLY a JSON object in this exact format:
{{
    "priority": <1, 2, or 3>,
    "reasoning": "<brief explanation considering content AND context>"
}}

Your response:"""
    
    return prompt


# Test prompt on sample email
sample_email = df_annotate.iloc[0]
test_prompt = create_annotation_prompt(sample_email)

print("Sample Annotation Prompt:")
print("="*80)
print(test_prompt[:1000] + "...")
print("="*80)

Sample Annotation Prompt:
You are an expert email prioritization assistant. Your task is to assign a priority level to an email based on its content AND the recipient's current context.

**Priority Levels:**
1 (Low): Informational content, no specific action required, can be deferred without consequences
2 (Normal): Standard business communication requiring timely but not immediate response (within hours/days)
3 (Critical): Immediate action required, blocks other work, or has severe consequences if delayed (within minutes/hour)

**Recipient Context:**
- Current workload: moderately busy with ongoing tasks and regular meetings
- Time of day: afternoon
- Number of recipients: 1

**Email Details:**
Subject: re:spreads

Body:
mkt getting a little more bearish the back of winter i think-if we get another
cold blast jan/feb mite move out. with oil moving down and march closer flat 
px
wide to jan im not so bearish these sprds now-less bullish march april as 
well.....

**Instructions:**
1. C

## 2.5 Annotation Function using Groq API

In [26]:
def annotate_email_with_api(email_data, model="openai/gpt-oss-120b", max_retries=3):
    """
    Annotate email using Groq API with OpenAI GPT-OSS-120B.
    
    Args:
        email_data: Row from DataFrame with email info
        model: Groq model to use (default: openai/gpt-oss-120b - 120B params, fast, cheap)
        max_retries: Maximum retry attempts
    
    Returns:
        Dict with priority and reasoning
    """
    prompt = create_annotation_prompt(email_data)
    
    for attempt in range(max_retries):
        try:
            # Use Groq with OpenAI GPT-OSS-120B
            response = groq_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert email prioritization assistant. You MUST respond with ONLY valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=800,  # Increased from 400 - gives more room for JSON completion
                response_format={"type": "json_object"}
            )
            
            # Parse response
            response_text = response.choices[0].message.content.strip()
            
            # Try to extract JSON
            if '{' in response_text and '}' in response_text:
                json_start = response_text.find('{')
                json_end = response_text.rfind('}') + 1
                json_text = response_text[json_start:json_end]
                result = json.loads(json_text)
                
                # Validate priority is 1, 2, or 3
                if result.get('priority') in [1, 2, 3]:
                    return result
                else:
                    print(f"Invalid priority: {result.get('priority')}. Retrying...")
            else:
                print(f"No JSON found. Retrying...")
                
        except json.JSONDecodeError as e:
            print(f"JSON error (attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(2)
        except Exception as e:
            error_msg = str(e)
            if "rate_limit" in error_msg.lower():
                print(f"Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
            else:
                print(f"Error (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(2)
    
    # If all retries fail, return default
    print(f"Failed after {max_retries} attempts. Using default.")
    return {
        'priority': 2,
        'reasoning': 'Failed to generate annotation'
    }


# Test on single email with GPT-OSS-120B
print("Testing annotation with OpenAI GPT-OSS-120B (120B params on Groq)...\n")
sample_annotation = annotate_email_with_api(sample_email, model="openai/gpt-oss-120b")
print(f"\nAnnotation result:")
print(f"Priority: {sample_annotation['priority']}")
print(f"Reasoning: {sample_annotation['reasoning']}")

# Show specs
print(f"\n🚀 GPT-OSS-120B Specs:")
print(f"  • 120B parameters (MoE architecture)")
print(f"  • 500+ tokens/sec on Groq")
print(f"  • $0.15/M input, $0.75/M output")
print(f"  • 131K context window")
print(f"  • Cost for 3,000 emails: ~$0.30 total")

Testing annotation with OpenAI GPT-OSS-120B (120B params on Groq)...


Annotation result:
Priority: 1
Reasoning: The email is an informal market commentary with no clear action request or deadline; given the recipient's moderate workload and lack of urgency, it is informational and can be deferred.

🚀 GPT-OSS-120B Specs:
  • 120B parameters (MoE architecture)
  • 500+ tokens/sec on Groq
  • $0.15/M input, $0.75/M output
  • 131K context window
  • Cost for 3,000 emails: ~$0.30 total


## 2.6 Batch Annotation

Annotate all selected emails. This will take some time depending on API rate limits.

In [27]:
def batch_annotate(df, model="openai/gpt-oss-120b", delay=1.0, checkpoint_freq=500):
    """
    Annotate emails in batches with checkpointing.
    
    Args:
        df: DataFrame with emails to annotate
        model: Model to use (default: openai/gpt-oss-120b)
        delay: Delay between API calls (seconds) - increased to avoid rate limits
        checkpoint_freq: Save checkpoint every N emails
    
    Returns:
        DataFrame with annotations
    """
    annotations = []
    
    print(f"Starting annotation of {len(df):,} emails using {model}...")
    print(f"Estimated time: ~{(len(df) * delay / 60):.1f} minutes")
    print(f"Estimated cost: ~${(len(df) * 0.0001):.2f}")
    print(f"Using {delay}s delay to respect free tier limits\n")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Annotate
        annotation = annotate_email_with_api(row, model=model)
        annotation['email_index'] = idx
        annotations.append(annotation)
        
        # Delay to respect rate limits
        time.sleep(delay)
        
        # Save checkpoint
        if (len(annotations) % checkpoint_freq == 0):
            checkpoint_df = pd.DataFrame(annotations)
            checkpoint_df.to_csv(f'annotations_checkpoint_{len(annotations)}.csv', index=False)
            print(f"\n✓ Checkpoint saved: {len(annotations)} annotations")
    
    # Convert to DataFrame
    annotations_df = pd.DataFrame(annotations)
    
    return annotations_df


# Run batch annotation with GPT-OSS-120B
print("=" * 80)
print("ANNOTATION TEST - First 50 emails with GPT-OSS-120B")
print("=" * 80)
print("\nUsing OpenAI GPT-OSS-120B with 1s delay (free tier safe)\n")

df_test_batch = df_annotate.head(50)
annotations_df = batch_annotate(df_test_batch, model="openai/gpt-oss-120b", delay=1.0)

print(f"\n✓ Annotation complete!")
print(f"\nPriority distribution:")
print(annotations_df['priority'].value_counts().sort_index())

# Calculate success rate
failed = (annotations_df['reasoning'] == 'Failed to generate annotation').sum()
success_rate = ((len(annotations_df) - failed) / len(annotations_df)) * 100
print(f"\nSuccess rate: {success_rate:.1f}% ({len(annotations_df) - failed}/{len(annotations_df)})")

if success_rate >= 95:
    print("\n✅ Excellent! Ready to annotate all 3,000 emails.")
    print("   Run: annotations_df = batch_annotate(df_annotate, model='openai/gpt-oss-120b', delay=1.0)")
    print("   This will take ~50 minutes but stay within free tier limits")
else:
    print(f"\n⚠️  Only {success_rate:.1f}% success rate. Check errors above.")

ANNOTATION TEST - First 50 emails with GPT-OSS-120B

Using OpenAI GPT-OSS-120B with 1s delay (free tier safe)

Starting annotation of 50 emails using openai/gpt-oss-120b...
Estimated time: ~0.8 minutes
Estimated cost: ~$0.01
Using 1.0s delay to respect free tier limits



100%|██████████| 50/50 [01:19<00:00,  1.58s/it]


✓ Annotation complete!

Priority distribution:
priority
1    27
2    19
3     4
Name: count, dtype: int64

Success rate: 100.0% (50/50)

✅ Excellent! Ready to annotate all 3,000 emails.
   Run: annotations_df = batch_annotate(df_annotate, model='openai/gpt-oss-120b', delay=1.0)
   This will take ~50 minutes but stay within free tier limits


## 2.7 Merge Annotations with Email Data

In [28]:
# Merge annotations with original data
df_annotated = df_test_batch.copy()
df_annotated['priority'] = annotations_df['priority'].values
df_annotated['priority_reasoning'] = annotations_df['reasoning'].values

print(f"Created annotated dataset with {len(df_annotated):,} emails")
print(f"\nPriority distribution:")
print(df_annotated['priority'].value_counts().sort_index())

# Show sample annotations
print("\n" + "="*80)
print("SAMPLE ANNOTATIONS")
print("="*80)

for priority in [1, 2, 3]:
    priority_emails = df_annotated[df_annotated['priority'] == priority]
    if len(priority_emails) > 0:
        sample = priority_emails.iloc[0]
        print(f"\nPriority {priority}:")
        print(f"Subject: {sample['subject'][:80]}")
        print(f"Context: {sample['workload_state']}, {sample['time_segment']}")
        print(f"Reasoning: {sample['priority_reasoning'][:150]}...")
        print("-" * 80)

Created annotated dataset with 50 emails

Priority distribution:
priority
1    27
2    19
3     4
Name: count, dtype: int64

SAMPLE ANNOTATIONS

Priority 1:
Subject: re:spreads
Context: busy, afternoon
Reasoning: The email is an informal market commentary without clear action items or deadlines; given the recipient's moderate workload, it can be treated as low‑...
--------------------------------------------------------------------------------

Priority 2:
Subject: Re: Hmmmmm........
Context: busy, morning
Reasoning: The email is an informal request for additional tickets with no explicit deadline or urgent language. It requires a response but does not block the re...
--------------------------------------------------------------------------------

Priority 3:
Subject: fyi:  TYZ/TYH ROLL SHOULD NARROW/Roll longs now
Context: very_busy, afternoon
Reasoning: The email recommends immediate market action (rolling positions now) to capture net carry; delay could cause financial loss, making

## 2.8 Save Annotated Dataset

In [29]:
# Save annotated dataset
output_file = f'enron_annotated_{len(df_annotated)}.csv'
df_annotated.to_csv(output_file, index=False)

print(f"✓ Saved annotated dataset to '{output_file}'")
print(f"\nDataset summary:")
print(f"  Total emails: {len(df_annotated):,}")
print(f"  Low priority (1): {(df_annotated['priority'] == 1).sum():,}")
print(f"  Normal priority (2): {(df_annotated['priority'] == 2).sum():,}")
print(f"  Critical priority (3): {(df_annotated['priority'] == 3).sum():,}")
print(f"\nReady for baseline modeling!")

✓ Saved annotated dataset to 'enron_annotated_50.csv'

Dataset summary:
  Total emails: 50
  Low priority (1): 27
  Normal priority (2): 19
  Critical priority (3): 4

Ready for baseline modeling!


In [30]:
# ============================================================================
# OPTIMIZED FULL BATCH ANNOTATION - PAID TIER (FAST MODE)
# Run this cell to annotate all 2,998 emails with your payment method
# ============================================================================

import datetime
import os

print("=" * 80)
print("FULL BATCH ANNOTATION - FAST MODE (Paid Tier)")
print("=" * 80)

# Check for existing checkpoint
checkpoint_files = [f for f in os.listdir('.') if f.startswith('annotations_checkpoint_')]
if checkpoint_files:
    latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
    checkpoint_df = pd.read_csv(latest_checkpoint)
    start_from = len(checkpoint_df)
    print(f"\n✓ Found checkpoint: {latest_checkpoint}")
    print(f"  Already annotated: {start_from} emails")
    print(f"  Resuming from email #{start_from + 1}")
    existing_annotations = checkpoint_df.to_dict('records')
else:
    start_from = 0
    existing_annotations = []
    print(f"\n  No checkpoint found. Starting from beginning.")

# Skip already annotated emails
remaining_df = df_annotate.iloc[start_from:]
remaining_count = len(remaining_df)

print(f"\n  Remaining to annotate: {remaining_count} emails")

# FAST settings for paid tier
DELAY = 0.2  # 5x faster than free tier!
CHECKPOINT_FREQ = 250  # Save progress every 250 emails

print(f"\n⚡ FAST MODE SETTINGS:")
print(f"  Delay between requests: {DELAY}s (was 1.0s on free tier)")
print(f"  Estimated time: ~{(remaining_count * DELAY / 60):.1f} minutes")
print(f"  Checkpoint frequency: Every {CHECKPOINT_FREQ} emails")
print(f"  Estimated cost: ${(remaining_count * 0.0001):.2f}")

# Confirm before starting
print(f"\n{'='*80}")
print(f"Ready to annotate {remaining_count} emails")
print(f"Starting in 3 seconds...")
print(f"{'='*80}\n")

import time as wait_time
wait_time.sleep(3)

# Start annotation
start_time = datetime.datetime.now()
print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

annotations = existing_annotations.copy()

for idx, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Annotating"):
    # Annotate
    annotation = annotate_email_with_api(row, model='openai/gpt-oss-120b')
    annotation['email_index'] = idx
    annotations.append(annotation)

    # Delay (much shorter with paid tier!)
    time.sleep(DELAY)

    # Save checkpoint
    if (len(annotations) % CHECKPOINT_FREQ == 0):
        checkpoint_df = pd.DataFrame(annotations)
        checkpoint_df.to_csv(f'annotations_checkpoint_{len(annotations)}.csv', index=False)
        elapsed = (datetime.datetime.now() - start_time).total_seconds() / 60
        rate = len(annotations) / elapsed if elapsed > 0 else 0
        remaining_time = ((len(df_annotate) - len(annotations)) / rate) if rate > 0 else 0
        print(f"\n✓ Checkpoint: {len(annotations)}/{len(df_annotate)} | "
              f"Elapsed: {elapsed:.1f}min | "
              f"ETA: {remaining_time:.1f}min | "
              f"Rate: {rate:.1f} emails/min")

# Convert to DataFrame
annotations_df_full = pd.DataFrame(annotations)

# End time tracking
end_time = datetime.datetime.now()
duration = end_time - start_time

print(f"\n" + "=" * 80)
print(f"✅ FULL ANNOTATION COMPLETE!")
print(f"=" * 80)
print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {duration}")
print(f"Average time per email: {(duration.total_seconds() / len(annotations_df_full)):.2f}s")
print(f"=" * 80)

# Show results
print(f"\n📊 PRIORITY DISTRIBUTION:")
print(annotations_df_full['priority'].value_counts().sort_index())

priority_counts = annotations_df_full['priority'].value_counts()
total = len(annotations_df_full)
print(f"\n📈 BREAKDOWN:")
print(f"  Low (1):      {priority_counts.get(1, 0):4d} ({priority_counts.get(1, 0)/total*100:.1f}%)")
print(f"  Normal (2):   {priority_counts.get(2, 0):4d} ({priority_counts.get(2, 0)/total*100:.1f}%)")
print(f"  Critical (3): {priority_counts.get(3, 0):4d} ({priority_counts.get(3, 0)/total*100:.1f}%)")

# Calculate success rate
failed = (annotations_df_full['reasoning'] == 'Failed to generate annotation').sum()
success_rate = ((len(annotations_df_full) - failed) / len(annotations_df_full)) * 100
print(f"\n✅ SUCCESS RATE: {success_rate:.1f}% ({len(annotations_df_full) - failed}/{len(annotations_df_full)})")

if failed > 0:
    print(f"⚠️  {failed} annotations failed - review errors above")
else:
    print(f"🎉 All {len(annotations_df_full)} annotations successful!")

# Save final checkpoint
annotations_df_full.to_csv('annotations_full_final.csv', index=False)
print(f"\n💾 Saved final annotations to: annotations_full_final.csv")

print(f"\n" + "=" * 80)
print(f"✅ READY FOR NEXT STEP: Run Cell 23 to merge with email data!")
print(f"=" * 80)

FULL BATCH ANNOTATION - FAST MODE (Paid Tier)

  No checkpoint found. Starting from beginning.

  Remaining to annotate: 2998 emails

⚡ FAST MODE SETTINGS:
  Delay between requests: 0.2s (was 1.0s on free tier)
  Estimated time: ~10.0 minutes
  Checkpoint frequency: Every 250 emails
  Estimated cost: $0.30

Ready to annotate 2998 emails
Starting in 3 seconds...

Start time: 2025-11-20 12:58:32



Annotating:   8%|▊         | 250/2998 [03:39<37:04,  1.24it/s]  


✓ Checkpoint: 250/2998 | Elapsed: 3.7min | ETA: 40.1min | Rate: 68.4 emails/min


Annotating:  17%|█▋        | 500/2998 [07:14<39:35,  1.05it/s]  


✓ Checkpoint: 500/2998 | Elapsed: 7.2min | ETA: 36.2min | Rate: 69.0 emails/min


...


✓ Checkpoint: 2500/2998 | Elapsed: 39.3min | ETA: 7.8min | Rate: 63.6 emails/min


Annotating:  92%|█████████▏| 2750/2998 [43:20<03:44,  1.10it/s]


✓ Checkpoint: 2750/2998 | Elapsed: 43.3min | ETA: 3.9min | Rate: 63.5 emails/min


Annotating: 100%|██████████| 2998/2998 [47:21<00:00,  1.06it/s]


✅ FULL ANNOTATION COMPLETE!
End time: 2025-11-20 13:45:54
Total duration: 0:47:21.193373
Average time per email: 0.95s

📊 PRIORITY DISTRIBUTION:
priority
1    1540
2    1305
3     153
Name: count, dtype: int64

📈 BREAKDOWN:
  Low (1):      1540 (51.4%)
  Normal (2):   1305 (43.5%)
  Critical (3):  153 (5.1%)

✅ SUCCESS RATE: 100.0% (2998/2998)
🎉 All 2998 annotations successful!

💾 Saved final annotations to: annotations_full_final.csv

✅ READY FOR NEXT STEP: Run Cell 23 to merge with email data!


## 🚀 FAST MODE - Full Annotation with Paid Tier

Now that you have payment set up, we can annotate MUCH faster:
- **Delay:** 0.2s (was 1.0s)
- **Estimated time:** ~10 minutes (was ~50 minutes)
- **Cost:** ~$0.30
- **Resumes from checkpoint** if available

## 2.9 Next Steps

To complete full annotation:
1. **Expand to full 3,000+ emails:** Run `batch_annotate(df_annotate, delay=0.5)`
2. **Validate annotation quality:** Manual review of random samples
3. **Calculate Cohen's Kappa:** Compare subset of manual vs. API annotations
4. **Refine prompts:** Based on validation results
5. **Proceed to baseline models:** Once we have 2,000+ annotations

---

**Current Status:** Test batch of 50 emails annotated successfully! 
**Next:** Expand to full 3,000-5,000 email annotation

## 2.10 Full Batch Annotation - 3,000 Emails

Now running the full annotation on all 2,998 emails.

**Specifications:**
- Model: OpenAI GPT-OSS-120B (120B params on Groq)
- Delay: 1.0s between requests (free tier safe)
- Checkpoints: Every 500 emails
- Estimated time: ~50 minutes
- Estimated cost: ~$0.30

**Note:** This will automatically save checkpoints every 500 emails in case of interruption.

In [ ]:
# Run full batch annotation on all 2,998 emails
print("=" * 80)
print("FULL BATCH ANNOTATION - 2,998 Emails with GPT-OSS-120B")
print("=" * 80)
print("\nStarting full annotation run...")
print("This will take approximately 50 minutes with 1s delay between requests.")
print("Checkpoints will be saved every 500 emails.\n")

# Start time tracking
import datetime
start_time = datetime.datetime.now()
print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

# Run batch annotation on all emails
annotations_df_full = batch_annotate(df_annotate, model='openai/gpt-oss-120b', delay=1.0, checkpoint_freq=500)

# End time tracking
end_time = datetime.datetime.now()
duration = end_time - start_time
print(f"\n" + "=" * 80)
print(f"✓ Full annotation complete!")
print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {duration}")
print(f"=" * 80)

# Show results
print(f"\nPriority distribution:")
print(annotations_df_full['priority'].value_counts().sort_index())

# Calculate statistics
priority_counts = annotations_df_full['priority'].value_counts()
total = len(annotations_df_full)
print(f"\nPriority percentages:")
print(f"  Low (1):      {priority_counts.get(1, 0):4d} ({priority_counts.get(1, 0)/total*100:.1f}%)")
print(f"  Normal (2):   {priority_counts.get(2, 0):4d} ({priority_counts.get(2, 0)/total*100:.1f}%)")
print(f"  Critical (3): {priority_counts.get(3, 0):4d} ({priority_counts.get(3, 0)/total*100:.1f}%)")

# Calculate success rate
failed = (annotations_df_full['reasoning'] == 'Failed to generate annotation').sum()
success_rate = ((len(annotations_df_full) - failed) / len(annotations_df_full)) * 100
print(f"\nSuccess rate: {success_rate:.1f}% ({len(annotations_df_full) - failed}/{len(annotations_df_full)})")

if failed > 0:
    print(f"⚠️  {failed} annotations failed - review errors above")
else:
    print(f"✅ All {len(annotations_df_full)} annotations successful!")

## 2.11 Merge Full Annotations with Email Data

After running the full annotation, merge the results with the original email data.

In [31]:
# Merge full annotations with original data
df_annotated_full = df_annotate.copy()
df_annotated_full['priority'] = annotations_df_full['priority'].values
df_annotated_full['priority_reasoning'] = annotations_df_full['reasoning'].values

print(f"Created full annotated dataset with {len(df_annotated_full):,} emails")
print(f"\nColumns: {list(df_annotated_full.columns)}")

# Verify data integrity
print(f"\nData integrity check:")
print(f"  Missing priorities: {df_annotated_full['priority'].isna().sum()}")
print(f"  Missing reasoning: {df_annotated_full['priority_reasoning'].isna().sum()}")
print(f"  Invalid priorities: {(~df_annotated_full['priority'].isin([1, 2, 3])).sum()}")

# Show sample annotations from each priority level
print("\n" + "="*80)
print("SAMPLE ANNOTATIONS FROM FULL DATASET")
print("="*80)

for priority in [1, 2, 3]:
    priority_emails = df_annotated_full[df_annotated_full['priority'] == priority]
    if len(priority_emails) > 0:
        # Show 2 samples per priority level
        samples = priority_emails.sample(n=min(2, len(priority_emails)), random_state=42)
        for idx, sample in samples.iterrows():
            print(f"\n{'='*80}")
            print(f"Priority {priority} - Sample")
            print(f"{'='*80}")
            print(f"From: {sample['from']}")
            print(f"Subject: {sample['subject'][:100] if pd.notna(sample['subject']) else '(no subject)'}")
            print(f"Context: {sample['workload_state']}, {sample['time_segment']}, {sample['recipient_count']} recipient(s)")
            print(f"Body preview: {str(sample['body'])[:200] if pd.notna(sample['body']) else '(no body)'}...")
            print(f"\nReasoning: {sample['priority_reasoning']}")
            print(f"{'='*80}")

Created full annotated dataset with 2,998 emails

Columns: ['file', 'datetime', 'from', 'to', 'subject', 'body', 'hour', 'day_of_week', 'subject_length', 'body_length', 'recipient_count', 'has_urgency_in_subject', 'has_urgency_in_body', 'workload_state', 'time_segment', 'priority', 'priority_reasoning']

Data integrity check:
  Missing priorities: 0
  Missing reasoning: 0
  Invalid priorities: 0

SAMPLE ANNOTATIONS FROM FULL DATASET

Priority 1 - Sample
From: john.arnold@enron.com
Subject: RE:
Context: busy, afternoon, 1 recipient(s)
Body preview: he was contacted directly.  Why, are you trying to find a good one?

 -----Original Message-----
From: 	Slone, Jeanie  
Sent:	Monday, October 29, 2001 9:24 AM
To:	Arnold, John
Subject:	

Do you know w...

Reasoning: The email is a casual inquiry about a headhunter with no deadline or urgent action required; given the recipient's moderately busy afternoon workload, it can be deferred without consequence.

Priority 1 - Sample
From: phillip.alle

## 2.12 Save Full Annotated Dataset & Generate Summary Statistics

In [32]:
# Save full annotated dataset
output_file_full = f'enron_annotated_{len(df_annotated_full)}.csv'
df_annotated_full.to_csv(output_file_full, index=False)

print(f"✓ Saved full annotated dataset to '{output_file_full}'")
print(f"\nFile size: {os.path.getsize(output_file_full) / (1024*1024):.2f} MB")

# Generate comprehensive summary statistics
print("\n" + "="*80)
print("FULL ANNOTATION SUMMARY")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"  Total emails annotated: {len(df_annotated_full):,}")
print(f"  Date range: {pd.to_datetime(df_annotated_full['datetime']).min()} to {pd.to_datetime(df_annotated_full['datetime']).max()}")
print(f"  Unique senders: {df_annotated_full['from'].nunique():,}")

print(f"\n🎯 Priority Distribution:")
priority_counts = df_annotated_full['priority'].value_counts().sort_index()
for priority in [1, 2, 3]:
    count = priority_counts.get(priority, 0)
    pct = (count / len(df_annotated_full)) * 100
    priority_name = {1: 'Low', 2: 'Normal', 3: 'Critical'}[priority]
    bar = '█' * int(pct / 2)  # Visual bar
    print(f"  {priority_name:8s} ({priority}): {count:4d} ({pct:5.1f}%) {bar}")

print(f"\n⏰ Priority by Workload State:")
workload_priority = pd.crosstab(df_annotated_full['workload_state'], df_annotated_full['priority'], normalize='index') * 100
print(workload_priority.round(1).to_string())

print(f"\n🕐 Priority by Time Segment:")
time_priority = pd.crosstab(df_annotated_full['time_segment'], df_annotated_full['priority'], normalize='index') * 100
print(time_priority.round(1).to_string())

print(f"\n🚨 Priority by Urgency Keywords:")
urgency_priority = pd.crosstab(
    df_annotated_full['has_urgency_in_subject'] | df_annotated_full['has_urgency_in_body'], 
    df_annotated_full['priority'], 
    normalize='index'
) * 100
urgency_priority.index = ['No urgency keywords', 'Has urgency keywords']
print(urgency_priority.round(1).to_string())

print(f"\n✅ Annotation Quality:")
failed_count = (df_annotated_full['priority_reasoning'] == 'Failed to generate annotation').sum()
success_rate = ((len(df_annotated_full) - failed_count) / len(df_annotated_full)) * 100
print(f"  Success rate: {success_rate:.1f}%")
print(f"  Failed annotations: {failed_count}")
print(f"  Average reasoning length: {df_annotated_full['priority_reasoning'].str.len().mean():.0f} characters")

print(f"\n💰 Cost & Performance:")
estimated_cost = len(df_annotated_full) * 0.0001
print(f"  Estimated cost: ${estimated_cost:.2f}")
print(f"  Model used: OpenAI GPT-OSS-120B (120B params)")

print("\n" + "="*80)
print("✅ ANNOTATION COMPLETE - Ready for baseline modeling!")
print("="*80)

✓ Saved full annotated dataset to 'enron_annotated_2998.csv'

File size: 7.31 MB

FULL ANNOTATION SUMMARY

📊 Dataset Statistics:
  Total emails annotated: 2,998
  Date range: 1980-01-01 00:00:00+00:00 to 2002-03-21 18:48:12+00:00
  Unique senders: 422

🎯 Priority Distribution:
  Low      (1): 1540 ( 51.4%) █████████████████████████
  Normal   (2): 1305 ( 43.5%) █████████████████████
  Critical (3):  153 (  5.1%) ██

⏰ Priority by Workload State:
priority           1     2    3
workload_state                 
busy            44.3  50.0  5.8
normal          51.6  42.8  5.6
quiet           50.9  45.2  3.9
very_busy       61.8  33.3  4.9

🕐 Priority by Time Segment:
priority         1     2    3
time_segment                 
afternoon     54.8  39.9  5.3
evening       51.6  45.2  3.2
morning       43.5  49.8  6.7
night         55.9  40.1  3.9

🚨 Priority by Urgency Keywords:
priority                 1     2     3
No urgency keywords   52.5  44.2   3.2
Has urgency keywords  46.8  40.6  12.5